# Solutions · 09 · Relaxation spectra

Worked solutions to the exercises of [notebook 09](../notebooks/09_relaxation_spectra.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, tts
from engrheo import viscoelastic as ve
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
melt = pd.read_csv(datasets.path("polystyrene_frequency_sweeps.csv"), comment="#")
curves = {T_C + 273.15: (g.omega_rad_s.to_numpy(), g[["G_storage_Pa", "G_loss_Pa"]].to_numpy()) for T_C, g in melt.groupby("temperature_C")}
mc = tts.master_curve(curves, 443.15)
w = np.concatenate([mc["omega_reduced"][T_] for T_ in sorted(curves)])
Y = np.vstack([mc["y"][T_] for T_ in sorted(curves)])
o = np.argsort(w); w, Gp, Gpp = w[o], Y[o, 0], Y[o, 1]
g_true = np.array([2.7e6, 1.08e6, 6.3e5, 5.4e5, 4.95e5, 3.6e5, 1.35e5]); tau_true = np.array([1e-4, 1e-3, 1e-2, 1e-1, 1.0, 5.0, 20.0])

## Exercise 1

Remove all data below 1 rad/s (reduced frequency) and refit. How do the fitted zero-shear viscosity and steady-state compliance change, and why?

In [3]:
full = ve.fit_spectrum(w, Gp, Gpp)
m = w >= 1.0
cut = ve.fit_spectrum(w[m], Gp[m], Gpp[m])
for name, f in (("all data", full), ("only omega >= 1 rad/s", cut)):
    print(f"{name:<22}: eta0 = {f.eta0:.3g} Pa s, Je0 = {ve.steady_state_compliance(f.g, f.tau):.3g} 1/Pa")
print(f"true: eta0 = {ve.zero_shear_viscosity(g_true, tau_true):.3g} Pa s, Je0 = {ve.steady_state_compliance(g_true, tau_true):.3g} 1/Pa")

all data              : eta0 = 5e+06 Pa s, Je0 = 2.54e-06 1/Pa
only omega >= 1 rad/s : eta0 = 4.64e+06 Pa s, Je0 = 1.82e-06 1/Pa
true: eta0 = 5.06e+06 Pa s, Je0 = 2.48e-06 1/Pa


Without the terminal zone the longest relaxation times are unknown: the zero-shear viscosity comes out about
8 % low and the steady-state compliance - which depends even more strongly on the slowest modes - more than
25 % low, although the fit to the remaining data is excellent. Integral properties are robust only if the data
reach the terminal region.

## Exercise 2

Fit spectra with 1, 2, 5 and 10 modes per decade. How does the misfit change, and from which density on does adding modes stop helping?

In [4]:
decades = np.log10(1 / w.min()) + 1 - (np.log10(1 / w.max()) - 1)
for per in (1, 2, 5, 10):
    f = ve.fit_spectrum(w, Gp, Gpp, n_modes=int(np.ceil(per * decades)))
    print(f"{per:2d} modes per decade: rms misfit {100*f.rel_rms:.1f} %")

 1 modes per decade: rms misfit 8.3 %
 2 modes per decade: rms misfit 2.7 %
 5 modes per decade: rms misfit 2.1 %
10 modes per decade: rms misfit 2.0 %


One mode per decade leaves a visibly rippled fit; the misfit falls steeply up to two modes per decade and
reaches the noise level of the data (2 %) at about five. Beyond that, extra modes only make the (already
ill-posed) spectrum less unique - there is nothing left to fit.

## Exercise 3

**Implement it yourself:** compute the steady-state compliance $J_e^0 = \sum g_i\tau_i^2/(\sum g_i\tau_i)^2$ from the fitted spectrum by hand, and check it against the long-time behaviour of the creep compliance from `ve.creep_from_relaxation` ($J(t) \to J_e^0 + t/\eta_0$).

In [5]:
f = ve.fit_spectrum(w, Gp, Gpp)
Je0 = np.sum(f.g * f.tau**2) / np.sum(f.g * f.tau) ** 2
print(f"longest fitted relaxation time with g > 0: {f.tau[f.g > 0].max():.0f} s")
t = np.r_[0.0, np.logspace(-6, 6, 2000)]                      # run far beyond the slowest mode
J = ve.creep_from_spectrum(f.g, f.tau, t)
late = t > 1e5
slope, icpt = np.polyfit(t[late], J[late], 1)
print(f"Je0 from the spectrum {Je0:.4g} 1/Pa; creep intercept {icpt:.4g} 1/Pa; 1/eta0: {1/f.eta0:.4g}, creep slope {slope:.4g}")

longest fitted relaxation time with g > 0: 3195 s


Je0 from the spectrum 2.544e-06 1/Pa; creep intercept 2.543e-06 1/Pa; 1/eta0: 2.001e-07, creep slope 2.001e-07


At long times the creep compliance is a straight line $J_e^0 + t/\eta_0$: its intercept is the steady-state
compliance and its slope the inverse zero-shear viscosity, both matching the formulas from the spectrum. "Long"
means long compared with the *slowest* mode - here the fitted spectrum contains a mode of about an hour, fitted
to the lowest-frequency data, so the straight line is reached only after days of creep.